In [34]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator #Genera datos (imagenes) sintéticos
from keras import optimizers #Técnica del descenso del gradiente, sirve para minimizar el error en el proceso de entrenamiento
from keras.models import Sequential # Estacebe un modelo de red neuronal por capas
from keras.layers import Dense, Flatten, Dropout, Activation
#Dense = define las neuronas por cada capa
#Flatter = aplana los datos en formato matriz N-Tensor --> 1-Tensor
#Dropout = técnica de reducción del sobreajuste o sobre entrenamiento (apagar % de neuronas)
#Activation = determina las funciones de activiación en cada neurona (Relu, sigmoid, softmax, tanh, linear, etc)
from keras.layers import Convolution2D, MaxPooling2D


In [ ]:
#Definir los hiperparametros de la red neurona convolucional (Convolutional Neural Network - CNN)

#Definir la ruta de los datos de entranamiento
entrenar = "CNN_Imagenes/entrenar"
validar = "CNN_Imagenes/validar"

#Hiperparametros
epocas = 100
altura,anchura = 400,400
batch_size = 2
pasos = 100

#Definir la cantidad de kernels por cada capa
kernel1=32 # 2,4,8,16,32,64,128,256, 512, etc
kernel1_size = (3,3)
kernel2=64
kernel2_size = (4,4)
size_pooling = (3,3)
clases = 2 #Numero de objetos a detectar o identificar




In [36]:
#Generar datos sintéticos (se recomienda si la cantidad de datos es pequeña)

entrenamiento = ImageDataGenerator(rescale=1/255,
                             zoom_range=0.2,
                             horizontal_flip=True)

validacion = ImageDataGenerator(rescale=1/255)

#Extraer las imagenes de las carpetas

imagenes_entrenamiento = entrenamiento.flow_from_directory(entrenar,
                                                      target_size=(anchura,altura),
                                                      batch_size=batch_size,
                                                      class_mode="categorical")
imagenes_validacion = validacion.flow_from_directory(validar,
                                                  target_size=(anchura,altura),
                                                  batch_size=batch_size,
                                                  class_mode="categorical")

Found 1000 images belonging to 2 classes.
Found 300 images belonging to 2 classes.


In [38]:
#Definir la arquitectura de la red neuronal convolucional

CNN = Sequential()

CNN.add(Convolution2D(kernel1,
                      kernel1_size,
                      padding="same",
                      input_shape=(altura,anchura,3),
                      activation="relu")) #Primera capa convolucional
CNN.add(MaxPooling2D(pool_size=size_pooling)) #Capa de submuestreo 

CNN.add(Convolution2D(kernel2,
                      kernel2_size,
                      padding="same",
                      input_shape=(altura,anchura,3),
                      activation="relu")) #Segunda capa convolucional
CNN.add(MaxPooling2D(pool_size=size_pooling)) #Capa de submuestreo 

#Aplanar las matrices en formato de vector
CNN.add(Flatten())

#Conectar a el perceptrón multicapa
CNN.add(Dense(255,activation="relu"))
CNN.add(Dense(255,activation="relu"))
CNN.add(Dense(255,activation="relu"))
CNN.add(Dropout(0.5))

#Definir la capa de salida
CNN.add(Dense(clases,activation="softmax"))


In [ ]:
#Definir los parametros del entrenamiento
CNN.compile(loss="categorical_crossentropy",optimizer="adam",metrics=["acc","mse"])

In [40]:
#Realizamos el entrenamiento
historico = CNN.fit(imagenes_entrenamiento,
                    validation_data=imagenes_validacion,
                    epochs=epocas,
                    validation_steps=pasos,
                    verbose=1)

Epoch 1/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 89s 173ms/step - acc: 0.5740 - loss: 0.7227 - mse: 0.2365 - val_acc: 0.6550 - val_loss: 0.6515 - val_mse: 0.2302
Epoch 2/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 102s 203ms/step - acc: 0.6330 - loss: 0.6604 - mse: 0.2188 - val_acc: 0.5000 - val_loss: 6.1755 - val_mse: 0.4998
Epoch 3/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 117s 233ms/step - acc: 0.6820 - loss: 0.6401 - mse: 0.1987 - val_acc: 0.6250 - val_loss: 0.9015 - val_mse: 0.2431
Epoch 4/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 153s 306ms/step - acc: 0.7840 - loss: 0.5187 - mse: 0.1531 - val_acc: 0.5850 - val_loss: 0.8848 - val_mse: 0.2591
Epoch 5/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 170s 339ms/step - acc: 0.8480 - loss: 0.3648 - mse: 0.1119 - val_acc: 0.7700 - val_loss: 0.6480 - val_mse: 0.1666
Epoch 6/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 179s 357ms/step - acc: 0.8550 - loss: 0.3525 - mse: 0.1070 - val_acc: 0.7900 - val_loss: 0.5175 - val_mse: 0.1555
Epoch 7/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 182s 363ms/step - acc: 0.8780 

In [41]:
#Guarda el modelo entrenado
CNN.save("CNN_Imagenes/Modelo/cnn.h5")
CNN.save_weights("CNN_Imagenes/Modelo/cnn_pesos.weights.h5")

In [ ]:
# Evaluacion del modelo entrenado con logica de decision CNN + OCR

import os  # Libreria para rutas y variables del sistema
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Oculta mensajes informativos de TensorFlow
import numpy as np  # Libreria numerica para arreglos y operaciones
from tensorflow.keras.utils import load_img, img_to_array  # Carga imagen y la convierte a arreglo
from keras.models import load_model  # Carga el modelo entrenado desde archivo
import cv2  # OpenCV para procesamiento de imagen
import easyocr  # Libreria OCR para leer texto en imagenes
import ssl  # Manejo de contexto SSL
import warnings  # Manejo de advertencias

warnings.filterwarnings('ignore')  # Oculta advertencias para una salida limpia
ssl._create_default_https_context = ssl._create_unverified_context  # Evita errores SSL en algunas maquinas

# Imagenes de prueba
ImagenNegativa1 = "CNN_Imagenes/validar/negativos/JOSE_lejos_19_jpg.rf.ec73fb67cce03749ef2b46f7714ece31.jpg"  # Ejemplo sin placa 1
ImagenNegativa2 = "CNN_Imagenes/validar/negativos/NO20250404-103115-000128F_mp4-0126_jpg.rf.e02ab587cc80c120ea4c3f66dc0c4884.jpg"  # Ejemplo sin placa 2
ImagenNegativa3 = "CNN_Imagenes/validar/negativos/images19_jpg.rf.7974ef0571a2a864dfabb739805fea86.jpg"  # Ejemplo sin placa 3
ImagenNegativa4 = "CNN_Imagenes/validar/negativos/images205_jpg.rf.9e486b8454b136cc6f009ae6f23d64f8.jpg"  # Ejemplo sin placa 4
ImagenNegativa5 = "CNN_Imagenes/validar/negativos/prohibido_circular_motos_jpg.rf.2d4d8fa436abad7a401af117296ae26f.jpg"  # Ejemplo sin placa 5

ImagenPlaca1 = "CNN_Imagenes/validar/placas/treino (406)_jpg.rf.OqtOeWYyJrJsOubinjkP.jpg"  # Ejemplo con placa 1
ImagenPlaca2 = "CNN_Imagenes/validar/placas/treino (497)_jpg.rf.z5jMvLQjZ26s5Cikb2aW.jpg"  # Ejemplo con placa 2
ImagenPlaca3 = "CNN_Imagenes/validar/placas/treino (459)_jpg.rf.qJUAXHMYAkHrP6YM33kx.jpg"  # Ejemplo con placa 3
ImagenPlaca4 = "CNN_Imagenes/validar/placas/82ae72b4699e1ebf2_jpg.rf.36c0ee5b2bd8ce7a92fc99a263f0b58f.jpg"  # Ejemplo con placa 4
ImagenPlaca5 = "CNN_Imagenes/validar/placas/treino (395)_jpg.rf.T5rONUZw5vRYKe28burx.jpg"  # Ejemplo con placa 5

# 1. Configuracion
ruta_imagen = ImagenNegativa4  # Selecciona que imagen se analizara
altura, anchura = 400, 400  # Tamano de entrada esperado por la CNN
modelo_path = "CNN_Imagenes/Modelo/cnn.h5"  # Ruta del archivo del modelo
pesos_path = "CNN_Imagenes/Modelo/cnn_pesos.weights.h5"  # Ruta de los pesos entrenados

# 2. Cargar modelos
print("Analizando la imagen...")  # Mensaje de inicio
reader = easyocr.Reader(['en'], gpu=False, verbose=False)  # Inicializa OCR en CPU y en ingles
cnn = load_model(modelo_path, compile=False)  # Carga arquitectura del modelo
cnn.load_weights(pesos_path)  # Carga los pesos al modelo

if os.path.exists(ruta_imagen):  # Verifica que la imagen exista
    # 3. Prediccion de la CNN
    img_cnn = load_img(ruta_imagen, target_size=(anchura, altura))  # Carga y redimensiona la imagen
    img_cnn = img_to_array(img_cnn) / 255.0  # Convierte a arreglo y normaliza pixeles a 0-1
    img_cnn = np.expand_dims(img_cnn, axis=0)  # Agrega dimension batch para la red
    clase_pred = cnn.predict(img_cnn, verbose=0)  # Obtiene probabilidades de clase
    arg_max = np.argmax(clase_pred[0])  # Toma el indice de la clase con mayor probabilidad
    prob_cnn = clase_pred[0][arg_max]  # Toma la probabilidad de la clase ganadora

    print("\n" + "="*40)  # Separador visual
    print("ANALISIS DE LA CNN:")  # Titulo del bloque
    print("="*40)  # Separador visual
    print(f"Confianza de la CNN: {prob_cnn*100:.2f}%")  # Muestra confianza de la prediccion

    if arg_max == 1:  # Si la clase predicha es placa
        print("ESTADO: PLACA DETECTADA. Iniciando lectura OCR...")  # Mensaje de estado

        # 4. Solo si la CNN aprobo
        img_cv = cv2.imread(ruta_imagen)  # Carga imagen con OpenCV
        img_res = cv2.resize(img_cv, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)  # Hace zoom para mejorar OCR
        gray = cv2.cvtColor(img_res, cv2.COLOR_BGR2GRAY)  # Convierte a escala de grises
        resultados = reader.readtext(gray)  # Ejecuta OCR sobre la imagen

        if not resultados:  # Si OCR no encontro texto
            print("RESULTADO: No se pudo leer el texto claramente.")  # Mensaje de no lectura
        else:  # Si OCR encontro texto
            for (bbox, texto, probabilidad) in resultados:  # Recorre cada texto detectado
                if probabilidad > 0.15:  # Filtra por confianza minima
                    print(f"MATRICULA: {texto.upper()} (Confianza OCR: {probabilidad*100:.2f}%)")  # Imprime texto y confianza OCR
    else:  # Si la CNN no detecto placa
        print("ESTADO: PLACA NO DETECTADA. Lectura cancelada.")  # Cancela OCR

    print("="*40)  # Cierre visual
else:  # Si la ruta no existe
    print(f"Error: No existe el archivo {ruta_imagen}")  # Muestra error de archivo no encontrado


Analizando la imagen...

ANALISIS DE LA CNN:
Confianza de la CNN: 78.21%
ESTADO: PLACA NO DETECTADA. Lectura cancelada.


In [ ]:
import numpy as np  # Libreria numerica para arreglos y operaciones
from tensorflow.keras.utils import load_img, img_to_array  # Herramientas para manejo de imagen en Keras
from keras.models import load_model  # Carga de modelo entrenado
import os.path  # Utilidades de rutas
import cv2  # OpenCV para procesamiento de imagen
import easyocr  # OCR para leer texto en imagen
import ssl  # Manejo de contexto SSL

# Configuracion para Mac
ssl._create_default_https_context = ssl._create_unverified_context  # Evita error SSL en algunas descargas

# Inicializar el lector fuera de la funcion para mayor velocidad
reader = easyocr.Reader(['en'], gpu=False, verbose=False)  # Crea lector OCR una sola vez

def evaluar(imagen):  # Funcion que recibe un frame o imagen para analizar
    # Valores de los hiperparametros
    altura, anchura = 400, 400  # Tamano de entrada del modelo
    modelo = "CNN_Imagenes/Modelo/cnn.h5"  # Ruta del modelo
    pesos = "CNN_Imagenes/Modelo/cnn_pesos.weights.h5"  # Ruta de pesos

    # 1. Cargar modelo
    cnn = load_model(modelo, compile=False)  # Carga arquitectura del modelo
    cnn.load_weights(pesos)  # Carga pesos entrenados

    # 2. Preprocesar para la CNN
    imagen_pre = cv2.resize(imagen, (anchura, altura))  # Ajusta tamano de imagen
    imagen_pre = imagen_pre / 255.0  # Normaliza pixeles al rango 0-1
    imagen_pre = img_to_array(imagen_pre)  # Convierte a arreglo compatible
    imagen_pre = np.expand_dims(imagen_pre, axis=0)  # Agrega dimension batch

    # 3. Prediccion
    clase = cnn.predict(imagen_pre, verbose=0)  # Predice probabilidades de clase
    arg_max = np.argmax(clase[0])  # Obtiene indice de clase con mayor probabilidad
    prob_cnn = clase[0][arg_max]  # Obtiene probabilidad de clase ganadora

    print("\n" + "="*40)  # Separador visual
    print("ANALISIS DE LA CNN:")  # Titulo del analisis
    print("="*40)  # Separador visual
    print(f"Confianza de la CNN: {prob_cnn*100:.2f}%")  # Muestra confianza de la red

    # --- LOGICA DE DECISION ---
    if arg_max == 1:  # Si detecta placa
        print("ESTADO: PLACA DETECTADA. Iniciando lectura OCR...")  # Mensaje de estado

        # 4. OCR con zoom para mejorar lectura de placas
        img_zoom = cv2.resize(imagen, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)  # Aumenta imagen para OCR
        gray = cv2.cvtColor(img_zoom, cv2.COLOR_BGR2GRAY)  # Convierte a grises
        resultados_ocr = reader.readtext(gray)  # Ejecuta OCR

        if not resultados_ocr:  # Si no hay texto detectado
            print("RESULTADO: No se pudo leer el texto claramente.")  # Mensaje de fallo
        else:  # Si se detecto texto
            for (bbox, texto, probabilidad) in resultados_ocr:  # Recorre resultados del OCR
                if probabilidad > 0.20:  # Umbral minimo de confianza
                    print(f"MATRICULA: {texto.upper()} (Confianza OCR: {probabilidad*100:.2f}%)")  # Imprime texto y confianza
    elif arg_max == 0:  # Si detecta no placa
        print("ESTADO: PLACA NO DETECTADA. Pudo haberse confundido.")  # Mensaje de estado negativo

    print("="*40)  # Cierre visual del reporte


In [10]:
import cv2

capture = cv2.VideoCapture(0)

while(1):
    _,frame = capture.read() #leemos cada frame de la camara
    cv2.imshow("Ventana", frame)
    c = cv2.waitKey(5) & 0xFF #Esperamos una tecla

    if (c==27):
        break

    if (c==99):
        evaluar(frame)

capture.release()
cv2.destroyAllWindows()
cv2.waitKey(1) # Fuerza el cierre de la ventana en Mac
cv2.waitKey(1)
cv2.waitKey(1)
cv2.waitKey(1)


ANALISIS DE LA CNN:
Confianza de la CNN: 76.06%
ESTADO: PLACA NO DETECTADA. Pudo haberse confundido.

ANALISIS DE LA CNN:
Confianza de la CNN: 90.10%
ESTADO: PLACA NO DETECTADA. Pudo haberse confundido.

ANALISIS DE LA CNN:
Confianza de la CNN: 100.00%
ESTADO: PLACA DETECTADA. Iniciando lectura OCR...
RESULTADO: No se pudo leer el texto claramente.


-1